<!-- nav -->
[← 8.14 · SystemVerilog Patterns](08_14_SystemVerilog_Patterns.ipynb) · [Home](../../README.md) · [8.16 · SystemVerilog Constraints →](08_16_SystemVerilog_Constraints.ipynb)

# 8.15 · SystemVerilog Assertions

SystemVerilog Assertions describe behavior over clock cycles (IEEE 1800 clause 16): a **sequence** says what happens
from one cycle to the next, `a ##1 b`, `a` then `b` a cycle later; a **property** says what must hold of sequences,
`a |=> b`, whenever `a`, then `b` in the next cycle. Tutorial 8 showed one of each (08.4). This page builds them up in
their dialect, `SystemVerilog Assertions`: delays and cycle ranges, repetitions and the sequence operators, the
property operators, a property's clock and reset, each in the builders' fluent calls beside the text SystemVerilog
writes, the printer placing parentheses by the standard's precedence. Then what is done with them: held by a call's
argument, validated, with one refused, and never evaluated, though the expressions among them are; last, the editions
that have each kind. The statement that asserts a property, `assert property (...)`, is mbse-programs'.

Like the rest of Part I, it starts from `dialects()`, a store with every dialect installed and defined. `name` and
`literal` write a name and a literal of SystemVerilog's `Expressions` (tutorials 3 and 4), which are a sequence's
simplest operands, and `refused` gives what an evaluator says of an
expression, its value or the reason it refuses it, as tutorial 8's does.

In [1]:
import assert from "node:assert/strict";
import * as SystemVerilog from "@mbse/expressions/SystemVerilog/Expressions";
import * as SVAssertions from "@mbse/expressions/SystemVerilog/Assertions";
import { Domains, Standards, Symbolics, Terms } from "@mbse/expressions/Framework";
import { dialects } from "./toolkit.js";

const store = dialects();
const S = store.Expressions;
const Assertions = SVAssertions.Expressions.DIALECT; // the dialect, as the framework reads it
const name = (spelling: string) => (d: any) => d.nameexpression((n: any) => n.name((a: any) => a.identifier((i: any) => i.spelling(spelling))));
const literal = (value: unknown) => (d: any) => d.literal((l: any) => l.value((v: any) => v.as_scalar(value)));
const [a, b, c, en, n] = [name("a"), name("b"), name("c"), name("en"), name("n")];


/** What `evaluate` says of `expression`: its value, or the reason it refuses it. */
function refused(evaluate: (expression: any) => unknown, expression: any): string {
  try {
    return String(evaluate(expression));
  } catch (error) {
    return (error as Error).message;
  }
}

<a id="T08.15.1"></a>
## 08.15.1 · A sequence of delays

```text
a ##1 b        a ##1 b ##[1:$] c        ##2 a
```

`a ##1 b` is a `delaysequence`: its `first` operand, `a`, then its `steps`, each a `DelayStep` helper holding a delay and
the sequence that follows it. A delay is an expression, `1`, or a cycle range, `[1:$]`, a `cyclerange` from one cycle to
any number (`$`, `Expressions`' `dollarexpression`); a sequence that starts with a delay has no `first`. Its simplest
operands are expressions: a name is a sequence that holds in the cycles where it is true. Its form gives its terms in
order, `first`, then each step's delay and sequence, two by two:

In [2]:
const delays: any[] = store.transaction(() => [
  (u: any) => u.delaysequence((s: any) => s.first(a).steps([(t: any) => t.delay(literal(1n)).sequence(b)])),
  (u: any) => u.delaysequence((s: any) => s.first(a).steps([
    (t: any) => t.delay(literal(1n)).sequence(b),
    (t: any) => t.delay((d: any) => d.cyclerange((r: any) => r.low(literal(1n)).high((d: any) => d.dollarexpression((r: any) => r)))).sequence(c)])),
  (u: any) => u.delaysequence((s: any) => s.steps([(t: any) => t.delay(literal(2n)).sequence(a)])),
].map((root) => S.SystemVerilog.Assertions.Expression((e: any) => e.root(root))));
for (const sequence of delays) {
  const form = Terms.form(Terms.term(sequence));
  console.log(SystemVerilog.Text.ToText(sequence).padEnd(19), form.kind, "| its terms:",
    form.arguments.map((t: any) => t != null ? Terms.kindOf(t)[0] : "-").join(", "));
}
assert.deepEqual(delays.map((s) => SystemVerilog.Text.ToText(s)), ["a ##1 b", "a ##1 b ##[1:$] c", "##2 a"]);

a ##1 b             delaysequence | its terms: nameexpression, literal, nameexpression


a ##1 b ##[1:$] c   delaysequence | its terms: nameexpression, literal, nameexpression, cyclerange, nameexpression


##2 a               delaysequence | its terms: -, literal, nameexpression


The empty `first` of `##2 a` is a `-` among its terms: a subterm left unset, as a lambda's `value` was in tutorial 7.

<a id="T08.15.2"></a>
## 08.15.2 · Repetitions and the sequence operators

```text
a[*2]    b[->1:3]    c[=n]    a and b    en throughout b    first_match(a or b)    a ##1 (b or c)
```

A repetition, `repetitionsequence`, repeats a sequence consecutively (`[*2]`), to its last match (`[->1:3]`, a goto
repetition) or not consecutively (`[=n]`), its count an expression or a cycle range. Two sequences combine by
`binarysequence`, its operator `and`, `or`, `intersect`, `within` or `throughout`; `first_match` keeps a sequence's
first match. Which operator binds tighter is the standard's (its Table 16-3), so a tree whose `##1` takes `b or c` as
its sequence prints parentheses the tree does not hold, as tutorial 5 showed for expressions:

In [3]:
const operators: any[] = store.transaction(() => [
  (u: any) => u.repetitionsequence((r: any) => r.sequence(a).operator("*").count(literal(2n))),
  (u: any) => u.repetitionsequence((r: any) => r.sequence(b).operator("->").count((d: any) => d.cyclerange((r: any) => r.low(literal(1n)).high(literal(3n))))),
  (u: any) => u.repetitionsequence((r: any) => r.sequence(c).operator("=").count(n)),
  (u: any) => u.binarysequence((s: any) => s.left(a).operator("and").right(b)),
  (u: any) => u.binarysequence((s: any) => s.left(en).operator("throughout").right(b)),
  (u: any) => u.firstmatchsequence((f: any) => f.sequence((u: any) => u.binarysequence((s: any) => s.left(a).operator("or").right(b)))),
  (u: any) => u.delaysequence((s: any) => s.first(a).steps([(t: any) => t.delay(literal(1n)).sequence((u: any) => u.binarysequence((s: any) => s.left(b).operator("or").right(c)))])),
].map((root) => S.SystemVerilog.Assertions.Expression((e: any) => e.root(root))));
for (const sequence of operators) {
  console.log(SystemVerilog.Text.ToText(sequence).padEnd(20), Terms.kindOf(Terms.term(sequence))[0]);
}
assert.deepEqual(operators.map((s) => SystemVerilog.Text.ToText(s)), [
  "a[*2]", "b[->1:3]", "c[=n]", "a and b", "en throughout b", "first_match(a or b)", "a ##1 (b or c)"]);

a[*2]                repetitionsequence


b[->1:3]             repetitionsequence


c[=n]                repetitionsequence


a and b              binarysequence


en throughout b      binarysequence


first_match(a or b)  firstmatchsequence


a ##1 (b or c)       delaysequence


<a id="T08.15.3"></a>
## 08.15.3 · Properties

```text
a |-> b    a |=> b    always a    s_eventually b    not (a and b)    nexttime [2] a    if (en) a else b
```

A property is built of sequences: an implication, `implicationproperty`, whose consequent must hold from the cycle its
antecedent matches (`|->`) or the next (`|=>`); a unary operator, `unaryproperty`, `not`, `always`, `s_eventually`,
`nexttime` and the rest, perhaps with a range of cycles; a binary one, `binaryproperty` (`and`, `or`, `implies`,
`until`, ...); and a conditional, `if (en) a else b`. Where a sequence may stand, so may a property, and an expression,
so a property's operands are this dialect's terms:

In [4]:
const properties: any[] = store.transaction(() => [
  (u: any) => u.implicationproperty((p: any) => p.antecedent(a).operator("|->").consequent(b)),
  (u: any) => u.implicationproperty((p: any) => p.antecedent(a).operator("|=>").consequent(b)),
  (u: any) => u.unaryproperty((p: any) => p.operator("always").operand(a)),
  (u: any) => u.unaryproperty((p: any) => p.operator("s_eventually").operand(b)),
  (u: any) => u.unaryproperty((p: any) => p.operator("not").operand((u: any) => u.binaryproperty((p: any) => p.left(a).operator("and").right(b)))),
  (u: any) => u.unaryproperty((p: any) => p.operator("nexttime").range(literal(2n)).operand(a)),
  (u: any) => u.conditionalproperty((p: any) => p.condition(en).consequence(a).alternative(b)),
].map((root) => S.SystemVerilog.Assertions.Expression((e: any) => e.root(root))));
for (const property of properties) {
  console.log(SystemVerilog.Text.ToText(property).padEnd(17), Terms.kindOf(Terms.term(property))[0]);
}
assert.deepEqual(properties.map((p) => SystemVerilog.Text.ToText(p)), [
  "a |-> b", "a |=> b", "always a", "s_eventually b", "not (a and b)", "nexttime [2] a", "if (en) a else b"]);

a |-> b           implicationproperty


a |=> b           implicationproperty


always a          unaryproperty


s_eventually b    unaryproperty


not (a and b)     unaryproperty


nexttime [2] a    unaryproperty


if (en) a else b  conditionalproperty


<a id="T08.15.4"></a>
## 08.15.4 · A property's clock and reset

```text
@(posedge clk) disable iff (rst) req |=> ack ##[1:3] done
```

A property is checked on a clock and abandoned while a reset holds: a `propertyspec` holds its `clock`, an
`EventControl` helper of `EventExpression`s (each an edge, an expression and perhaps an `iff` condition), its `disable`
condition, an expression, and its `property`. The clock is a helper, so the form gives it as a part; its signal is a
term all the same, and `Symbolics.free` finds every name the property reads, the clock's among them:

In [5]:
const handshake = store.transaction(() => S.SystemVerilog.Assertions.Expression((e: any) => e.root((u: any) => u.propertyspec((s: any) => s
  .clock((k: any) => k.events([(v: any) => v.edge("posedge").expression(name("clk"))]))
  .disable(name("rst"))
  .property((u: any) => u.implicationproperty((p: any) => p.antecedent(name("req")).operator("|=>").consequent((d: any) => d.delaysequence((q: any) => q.first(name("ack")).steps([
    (t: any) => t.delay((d: any) => d.cyclerange((r: any) => r.low(literal(1n)).high(literal(3n)))).sequence(name("done"))])))))))));
const form = Terms.form(Terms.term(handshake));
console.log(SystemVerilog.Text.ToText(handshake));
console.log(form.kind, "| clock:", (form.attributes.get("clock") as Terms.Part).kind, "| its terms:",
  form.arguments.map((t: any) => t != null ? Terms.kindOf(t)[0] : "-").join(", "));
console.log("free:", [...Symbolics.free(handshake, Assertions)].sort().join(", "));
assert.equal(SystemVerilog.Text.ToText(handshake), "@(posedge clk) disable iff (rst) req |=> ack ##[1:3] done");

@(posedge clk) disable iff (rst) req |=> ack ##[1:3] done


propertyspec | clock: eventcontrol | its terms: nameexpression, implicationproperty, nameexpression, -


free: ack, clk, done, req, rst


<a id="T08.15.5"></a>
## 08.15.5 · Held by a call, validated, and one refused

```text
f(a ##1 b)
```

A named sequence or property is given its arguments as a call is (16.8): a call's arguments, and a named connection's
value, hold this dialect's `Term`, the properties of `Expressions`' kinds read from the store as tutorial 8 did (08.3). So
`f(a ##1 b)` is `Expressions`' call, its argument an Assertions term, and its host validates it, reading it through the
dialects it reads. `validate` checks each kind's operator among its choices: `->` is no implication's, and is refused:

In [6]:
const union = store.registered(Assertions.union);
const host = store.registered(SystemVerilog.Expressions.DIALECT.union).branches as any[];
console.log(host.flatMap((b) => [...b.schema.properties.entries()]
  .filter(([, prop]: [string, any]) => prop.schema === union || prop.schema.element === union)
  .map(([p]: [string, any]) => `${b.name}.${p}`)).join(", "));
const called = store.transaction(() => S.SystemVerilog.Expressions.Expression((e: any) => e.root((u: any) => u.callexpression((f: any) => f
  .callee(name("f")).arguments([(d: any) => d.delaysequence((s: any) => s.first(a).steps([(t: any) => t.delay(literal(1n)).sequence(b)]))])))));
const argument = Terms.form(Terms.term(called)).arguments[1];
const dialect = SystemVerilog.Expressions.DIALECT;
console.log(SystemVerilog.Text.ToText(called), "| its argument:", Terms.kindOf(argument)[0], "of", dialect.owner(argument).name(),
  "| problems:", dialect.validate(called, ["f", "a"]).join("; ") || "none");
const arrow = store.transaction(() => S.SystemVerilog.Assertions.Expression((e: any) => e.root((u: any) => u.implicationproperty((p: any) => p.antecedent(a).operator("->").consequent(b)))));
console.log(SystemVerilog.Text.ToText(arrow), "|", Assertions.validate(arrow, ["a", "b"]).join("; "));
assert.deepEqual(Assertions.validate(arrow, ["a", "b"]), ["ImplicationProperty.operator cannot be '->'"]);

callexpression.arguments, namedconnection.value


f(a ##1 b) | its argument: delaysequence of SystemVerilog Assertions | problems: argument 0: sequence 0: nameexpression 'b' is not bound


a -> b | ImplicationProperty.operator cannot be '->'


Given `f` and `a` alone, the host reports the `b` the sequence reads, by where it stands: the call's argument 0, its
first step's sequence, `sequence 0`.

<a id="T08.15.6"></a>
## 08.15.6 · Never evaluated, but its expressions are

SystemVerilog's evaluator simulates no clock cycles: it refuses each of the dialect's kinds by name (stage 6, "Q11"),
held alone or as a call's argument, and an Assertions term has no domain to infer, `any`. An expression is no
assertion kind, though, and held by the same holder it still evaluates, `a + n` to a value of `integer` (tutorial 3,
03.5):

In [7]:
const evaluate = SystemVerilog.Evaluators.OfAny;
const values = { a: 1n, b: 0n, n: 2n };
for (const expression of [delays[0], operators[3], properties[1], handshake]) {
  console.log(SystemVerilog.Text.ToText(expression).padEnd(30), refused((e) => evaluate(e, values), expression));
}
const total = store.transaction(() => S.SystemVerilog.Assertions.Expression((e: any) => e.root((u: any) => u.binaryexpression((o: any) => o.left(a).operator("+").right(n)))));
console.log(SystemVerilog.Text.ToText(total).padEnd(30), String(evaluate(total, values)));
console.log("inferred:", Assertions.infer(delays[0], { a: Domains.Anything, b: Domains.Anything }).name());

a ##1 b                        SystemVerilog Assertions delaysequence is not evaluated


a and b                        SystemVerilog Assertions binarysequence is not evaluated


a |=> b                        SystemVerilog Assertions implicationproperty is not evaluated


@(posedge clk) disable iff (rst) req |=> ack ##[1:3] done SystemVerilog Assertions propertyspec is not evaluated


a + n                          32'sb00000000000000000000000000000011


inferred: any


<a id="T08.15.7"></a>
## 08.15.7 · Its editions are data

Assertions arrived with SystemVerilog, so the dialect covers IEEE 1800's five editions and none of Verilog's, each with
the kinds it has
([CAP-268: standards as tagged data](https://github.com/pitaman71/mbse-basis/blob/main/docs/capabilities/CAP-268.md)):
IEEE 1800-2009 added the strength (`strong(...)`, `weak(...)`), abort (`accept_on`, `reject_on`, ...) and case
properties:

In [8]:
const singleton = store.Instances.Expressions.SystemVerilog.Assertions;
const kinds = new Map((store.registered(Assertions.union).branches as any[]).map((branch) => [branch.schema, branch.name]));
for (const [designation, edition] of Standards.standards(singleton)) {
  console.log(designation.padEnd(19), Standards.tags(edition)[1]!.padEnd(19), Standards.kinds(edition).length, "kinds");
}
const [first, second] = ["SystemVerilog-2005", "SystemVerilog-2009"].map((label) => Standards.kinds(Standards.find(singleton, label)!));
console.log("added in IEEE 1800-2009:", second!.filter((k) => !first!.includes(k)).map((k) => kinds.get(k)).join(", "));

IEEE Std 1800-2005  SystemVerilog-2005  14 kinds


IEEE Std 1800-2009  SystemVerilog-2009  17 kinds


IEEE Std 1800-2012  SystemVerilog-2012  17 kinds


IEEE Std 1800-2017  SystemVerilog-2017  17 kinds


IEEE Std 1800-2023  SystemVerilog-2023  17 kinds


added in IEEE 1800-2009: strengthproperty, abortproperty, caseproperty


## What you learned

- **Sequences and properties are terms of `SystemVerilog Assertions`**: delays and cycle ranges, repetitions, the
  sequence and property operators, a property's clock and reset; their simplest operands are expressions.
- **The printer places parentheses by the standard's precedence**, from the tree, which holds none.
- **A call's arguments hold them**, so a named sequence or property is given one; its host validates the tree, an
  operator outside its kind's choices refused.
- **No clock cycle is simulated**: each kind is refused by name, while an expression among them still evaluates; its
  editions are IEEE 1800's, three kinds since IEEE 1800-2009.

Next, what a randomized object's values may be: [SystemVerilog Constraints](08_16_SystemVerilog_Constraints.ipynb).

<!-- nav -->
[← 8.14 · SystemVerilog Patterns](08_14_SystemVerilog_Patterns.ipynb) · [Home](../../README.md) · [8.16 · SystemVerilog Constraints →](08_16_SystemVerilog_Constraints.ipynb)